In [ ]:
import pandas as pd
import re

def read_news_to_dataframe(file_path):
    """
    读取新闻文本文件并将每篇新闻报道分隔开来，返回一个 DataFrame。
    
    :param file_path: 新闻文本文件路径
    :return: 包含每篇新闻报道的 DataFrame
    """
    # 读取文件内容
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()

    # 使用正则表达式按照一个或多个空行分隔新闻报道
    news_list = re.split(r'\n\s*\n', content.strip())  # \n\s*\n 匹配一个或多个空行

    # 创建 DataFrame
    df = pd.DataFrame(news_list, columns=["text"])

    return df

# 假设文件路径是 'news.txt'
file_path = '/hongyi/stream/PMI/dataset/News-Commentary.zh'

# 调用函数
df = read_news_to_dataframe(file_path)
df['labels'] = range(len(df))
# 打印 DataFrame
print(df.head())  # 打印前几行新闻报道


In [ ]:
import pandas as pd
import json

# 从文件中读取JSON数据
with open('/hongyi/stream/PMI/dataset/MultiUN_zh.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# 创建DataFrame，列名为'text'
df = pd.DataFrame(data, columns=['text'])
df['labels'] = range(len(df))
# 显示DataFrame
print(df.head())

In [ ]:

#本段落用时9min
from stream_topic.utils.dataset import TMDataset

# 创建 TMDataset 实例
dataset = TMDataset()#language="zh-cn", stopwords_path = '/hongyi/stream/stopwords/baidu_stopwords.txt'

# 假设你有一个名为 df 的 DataFrame，包含你的数据
# df = pd.DataFrame(...)

# 数据集名称
dataset_name = "news_zh"

# 指定保存数据集的目录
save_dir = "/hongyi/stream/PMI/dataset"
# 调用 create_load_save_dataset 方法来存储数据集
dataset.create_load_save_dataset(
    data=df,
    dataset_name=dataset_name,
    save_dir=save_dir,
    doc_column="text",  # 假设 DataFrame 中包含文本的列名为 "text_column"
    label_column="labels",  # 假设 DataFrame 中包含标签的列名为 "label_column"
    language = "chinese",
    stopwords_path = '/hongyi/stream/stopwords/baidu_stopwords.txt',
    min_word_length = 1
)#, min_word_freq=1

# 打印保存的数据集信息
print(dataset.dataframe.head())


In [ ]:
from stream_topic.models import KmeansTM,BERTopicTM,CBC,DCTE,NMFTM,SOMTM,CEDC,ETM,LDA,ProdLDA,SOMTM,NSTM,WordCluTM,CTM,TNTM,NeuralLDA,CTMNeg
from stream_topic.utils import TMDataset
#本段落用时9min
dataset = TMDataset(language="chinese", stopwords_path = '/hongyi/stream/stopwords/baidu_stopwords.txt')# 
dataset.fetch_dataset(name = "news_zh_jieba", dataset_path = "/hongyi/stream/PMI/dataset", source = 'local')
dataset.preprocess(model_type="KmeansTM", min_word_length = 1)
#本段落用时4h
model = KmeansTM(embedding_model_name="/hongyi/stream/sentence-transformers/Conan-embedding-v1/",stopwords_path = '/hongyi/stream/stopwords/baidu_stopwords.txt')#
model.fit(dataset,n_topics=10)
# model = NMFTM(stopwords_path = '/hongyi/stream/stopwords/scu_stopwords.txt')# 
# model.fit(dataset)#

topics = model.get_topics()
print(topics)

In [ ]:
from stream_topic.metrics import ISIM, INT, ISH,Expressivity, NPMI,PMI,cPMI, Embedding_Coherence, Embedding_Topic_Diversity
from sentence_transformers import SentenceTransformer
from stream_topic.metrics.metrics_config import MetricsConfig
MetricsConfig.set_PARAPHRASE_embedder("/hongyi/stream/sentence-transformers/Conan-embedding-v1/")#paraphrase-multilingual-mpnet-base-v2
MetricsConfig.set_SENTENCE_embedder("/hongyi/stream/sentence-transformers/Conan-embedding-v1/")

In [ ]:
import pandas as pd
def load_stopwords(stopwords_path):
        # load Chinese stopwords list
        return pd.read_csv(stopwords_path, names=['w'], sep='\t', encoding='UTF-8')
stopword = load_stopwords('/hongyi/stream/stopwords/baidu_stopwords.txt')

In [ ]:
metric = NPMI(dataset,language = "chinese", custom_stopwords=list(stopword)) #值越大越好   
scores = metric.score([["普京", "奥巴马"]])  #值越小越好
print("NPMI score:", scores)

In [ ]:
metric = PMI(dataset,language = "chinese", custom_stopwords=list(stopword)) #值越大越好   
scores = metric.score([["普京", "奥巴马"]])  #值越小越好
print("PMI score:", scores)

In [ ]:
def create_vocab_unsegmented(data, language,target_words=None, target_pairs=None):
        word_to_file = {}

        if language =="chinese":
            for file_num in range(0, len(data)):
                # 处理未分词数据，字符串匹配
                doc = data[file_num]
                doc = doc.strip()
                doc = re.sub(r"[^\u4e00-\u9fff\d]+", " ", doc)
                doc = re.sub(" +", " ", doc)
    
                if target_words:
                    # 遍历指定的目标词，检查每个目标词是否在文档中
                    for word in target_words:
                        # 使用字符串匹配检查目标词是否出现在文档中
                        if word in doc:
                            if word in word_to_file:
                                word_to_file[word].add(file_num)
                            else:
                                word_to_file[word] = {file_num}
                if target_pairs:
                    # 遍历指定的目标词对，检查每个目标词对是否在文档中
                    for pair in target_pairs:
                        word1, word2 = pair
                        if word1 in doc and word2 in doc:
                            if pair in word_to_file:
                                word_to_file[pair].add(file_num)
                            else:
                                word_to_file[pair] = {file_num}
        else:
            for file_num in range(0, len(data)):
                # 处理未分词数据，字符串匹配
                doc = data[file_num].lower()
                doc = doc.strip()
                doc = re.sub(r"[^a-zA-Z0-9]+\s*", " ", doc)
                doc = re.sub(" +", " ", doc)
    
                if target_words:
                    # 遍历指定的目标词，检查每个目标词是否在文档中
                    for word in target_words:
                        # 使用字符串匹配检查目标词是否出现在文档中
                        if word in doc:
                            if word in word_to_file:
                                word_to_file[word].add(file_num)
                            else:
                                word_to_file[word] = {file_num}
                if target_pairs:
                    # 遍历指定的目标词对，检查每个目标词对是否在文档中
                    for pair in target_pairs:
                        word1, word2 = pair
                        if word1 in doc and word2 in doc:
                            if pair in word_to_file:
                                word_to_file[pair].add(file_num)
                            else:
                                word_to_file[pair] = {file_num}

        return word_to_file

In [ ]:
import re
def read_news_to_dataframe(file_path):
    """
    读取新闻文本文件并将每篇新闻报道分隔开来，返回一个 DataFrame。
    
    :param file_path: 新闻文本文件路径
    :return: 包含每篇新闻报道的 DataFrame
    """
    # 读取文件内容
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()

    # 使用正则表达式按照一个或多个空行分隔新闻报道
    news_list = re.split(r'\n\s*\n', content.strip())  # \n\s*\n 匹配一个或多个空行

    # 创建 DataFrame
    df = pd.DataFrame(news_list, columns=["text"])

    return df

# 假设文件路径是 'news.txt'
file_path = '/hongyi/stream/PMI/dataset/News-Commentary.zh'

# 调用函数
df = read_news_to_dataframe(file_path)
df['labels'] = range(len(df))
target_words = ["普京", "奥巴马"]
target_pairs = [("普京", "奥巴马")]
words_count = create_vocab_unsegmented(df['text'], "chinese", target_words=target_words)
pair_count = create_vocab_unsegmented(df['text'], "chinese", target_pairs=target_pairs)

In [ ]:
import pandas as pd
import re
# 文件路径
file_path = "/hongyi/stream/dataset/cnews.val.txt"
# 创建一个空列表来存储处理后的数据
data = []
# 打开文件并读取每一行
with open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
        # 确保每一行至少有4个字符
        if len(line) > 3:
            # 提取前两个字符作为第一个元素
            first_two_chars = line[:2].strip()
            # 提取从第四个字开始的剩余部分作为第二个元素
            remaining_chars = line[3:].strip()
            # 将两个元素添加到列表中
            data.append([remaining_chars, first_two_chars])

# 创建 DataFrame
df = pd.DataFrame(data, columns=["text", "labels"])
target_words = ["普京", "奥巴马"]
target_pairs = [("普京", "奥巴马")]
words_count = create_vocab_unsegmented(df['text'], "chinese", target_words=target_words)
pair_count = create_vocab_unsegmented(df['text'], "chinese", target_pairs=target_pairs)

In [ ]:
import numpy as np
K = len(dataset.dataframe)
eps = 10 ** (-12)
pmi_w1w2 = np.log((len(pair_count[("普京", "奥巴马")]) * K) / ((len(words_count["普京"]) * len(words_count["奥巴马"])) + eps) + eps)
npmi_w1w2 = pmi_w1w2 / (-np.log((len(pair_count[("普京", "奥巴马")])) / K + eps))
print(pmi_w1w2)
print(npmi_w1w2)

In [ ]:
documents = list(dataset.dataframe.iloc[:,3:4]['text'].apply(lambda x: x.split()))
# documents = [item for sublist in documents for item in [sublist]*4]
K = len(documents)
l=[]
for doc in documents:
    l.append(len(doc))

In [ ]:
# 构建词索引映射
index_mappings = build_index_mapping(documents)

# 查询的词对
word_pairs = [("普京", "奥巴马")]

# 计算非重叠出现次数
result_f = count_nonoverlapping_intervals(index_mappings, word_pairs)

# 设置单词间隔阈值
threshold = 50
# 计算非重叠出现次数
result_f_hat = count_nonoverlapping_intervals_with_threshold(index_mappings, word_pairs, threshold)

# 打印结果
for pair, counts in result_f.items():
    f = counts
    # print(f"词对 {pair}: {counts}")
# 打印结果
for pair, counts in result_f_hat.items():
    f_hat = counts
    # print(f"词对 {pair}: {counts}")


In [ ]:
from collections import defaultdict

def build_index_mapping(documents):
    """
    构建词索引映射，用于快速查询每个词在文档中的位置。
    
    Args:
        documents (list of list): 数据集，每个文档是一个分词后的词列表。
    
    Returns:
        list of dict: 每个文档中词到位置列表的映射。
    """
    index_mappings = []
    for doc in documents:
        index_map = defaultdict(list)
        for i, word in enumerate(doc):
            index_map[word].append(i)
        index_mappings.append(index_map)
    return index_mappings


def count_nonoverlapping_intervals(index_mappings, word_pairs):
    """
    计算给定词对在每个文档中的非重叠出现次数。
    
    Args:
        index_mappings (list of dict): 每个文档的词索引映射。
        word_pairs (list of tuple): 要查询的词对列表，每个元素是 (word1, word2)。
    
    Returns:
        dict: 每个词对的非重叠出现次数列表。
    """
    results = {pair: [] for pair in word_pairs}
    
    for index_map in index_mappings:
        for word1, word2 in word_pairs:
            positions1 = sorted(index_map[word1])
            positions2 = sorted(index_map[word2])
            
            if not positions1 or not positions2:
                results[(word1, word2)].append(0)
                continue
            
            # 确定哪个词先出现
            if positions1[0] <= positions2[0]:
                positions_a, positions_b = positions1, positions2
            else:
                positions_a, positions_b = positions2, positions1
            
            # 计算非重叠区间
            count = 0
            last_start, last_end = -1, -1  # 上一个区间的起始和结束位置
            i, j = 0, 0  # i 是 A 的位置索引，j 是 B 的位置索引
            
            while i < len(positions_a) and j < len(positions_b):
                # 确定当前 A 和 B 的位置
                a, b = positions_a[i], positions_b[j]
                
                # A 和 B 的顺序决定新的区间
                if a < b:
                    if a > last_end and b > last_end:  # 不在上一个区间中
                        count += 1
                        last_start, last_end = a, b
                        i += 1
                        j += 1
                    else:
                        i += 1  # 跳过当前 A
                else:  # 如果 B 在 A 前面，调整顺序
                    if b > last_end and a > last_end:
                        count += 1
                        last_start, last_end = b, a
                        i += 1
                        j += 1
                    else:
                        j += 1  # 跳过当前 B
            
            results[(word1, word2)].append(count)
    
    return results


# # 示例数据集
# documents = [
#     ["我", "在", "处理", "大", "自然", "中", "使用", "自然", "语言", "处理", "方法", "来", "处理", "问题"],
#     ["我", "在", "处理", "大", "自然", "中", "使用", "自然", "语言", "处理", "方法"],
#     ["自然", "语言", "中", "使用", "处理", "方法", "自然", "处理", "语言"]
# ]

# # 构建词索引映射
# index_mappings = build_index_mapping(documents)

# # 查询的词对
# word_pairs = [("自然", "处理"),("处理", "自然")]

# # 计算非重叠出现次数
# results = count_nonoverlapping_intervals(index_mappings, word_pairs)

# # 打印结果
# for pair, counts in results.items():
#     print(f"词对 {pair}: {counts}")


In [ ]:
from collections import defaultdict

def count_nonoverlapping_intervals_with_threshold(index_mappings, word_pairs, threshold):
    """
    计算给定词对在每个文档中的非重叠出现次数，考虑间隔单词数阈值。

    Args:
        index_mappings (list of dict): 每个文档的词索引映射。
        word_pairs (list of tuple): 要查询的词对列表，每个元素是 (word1, word2)。
        threshold (int): A 和 B 之间允许的最大单词间隔。

    Returns:
        dict: 每个词对的非重叠出现次数列表。
    """
    results = {pair: [] for pair in word_pairs}

    for index_map in index_mappings:
        for word1, word2 in word_pairs:
            positions1 = sorted(index_map[word1])
            positions2 = sorted(index_map[word2])

            if not positions1 or not positions2:
                results[(word1, word2)].append(0)
                continue

            # 确定哪个词先出现
            if positions1[0] <= positions2[0]:
                positions_a, positions_b = positions1, positions2
            else:
                positions_a, positions_b = positions2, positions1

            # 计算非重叠区间
            count = 0
            last_start, last_end = -1, -1  # 上一个区间的起始和结束位置
            i, j = 0, 0  # i 是 A 的位置索引，j 是 B 的位置索引

            while i < len(positions_a) and j < len(positions_b):
                # 确定当前 A 和 B 的位置
                a, b = positions_a[i], positions_b[j]

                # 检查间隔是否满足阈值
                if abs(a - b) - 1 > threshold:
                    if a < b:
                        i += 1  # 跳过当前 A
                    else:
                        j += 1  # 跳过当前 B
                    continue

                # 确定 A 和 B 的顺序是否构成有效区间
                if a > last_end and b > last_end:  # 不在上一个区间中
                    count += 1
                    last_start, last_end = a, b
                    i += 1
                    j += 1
                else:
                    # 跳过被覆盖的 A 或 B
                    if a <= last_end:
                        i += 1
                    if b <= last_end:
                        j += 1

            results[(word1, word2)].append(count)

    return results


# # 示例数据集
# documents = [
#     ["我", "在", "大", "处理","我", "在", "大", "自然", "中", "使用", "自然", "语言", "处理", "方法", "来", "处理", "问题"],
#     ["我", "在", "大", "处理", "自然", "中", "使用", "自然", "语言", "处理", "方法"],
#     ["自然", "语言","我", "在", "大", "处理", "方法", "自然", "处理", "语言", "自然"]
# ]

# # 查询的词对
# word_pairs = [("自然", "处理"), ("处理", "自然")]

# # 设置单词间隔阈值
# threshold = 2

# # 计算非重叠出现次数
# results = count_nonoverlapping_intervals_with_threshold(index_mappings, word_pairs, threshold)

# # 打印结果
# for pair, counts in results.items():
#     print(f"词对 {pair}: {counts}")


In [ ]:
import numpy as np
def compute_hist_optimized(f, l, x, memo=None):
    """
    优化后的 ComputeHist 算法，利用动态规划加速
    :param f: 嵌入的非重叠出现次数
    :param l: 文档长度
    :param x: 跨度限制
    :param memo: 用于缓存中间结果的字典
    :return: 直方图分布 hist_f_l
    """
    if memo is None:
        memo = {}

    # 如果已经计算过，直接返回缓存结果
    if (f, l) in memo:
        return memo[(f, l)]

    # 初始化直方图 hist_f_l
    hist_f_l = np.zeros(f + 1, dtype=np.float64)

    # 边界条件
    if f > l:
        return hist_f_l  # 文档太短，无法嵌入
    if f == 0:
        hist_f_l[0] = 1  # 没有嵌入的可能
        memo[(f, l)] = hist_f_l
        return hist_f_l

    # 遍历文档中每个可能的 (i, j) 对
    for i in range(1,l):  # 从 1 到 l-1
        for j in range(i + 1, l+1):  # 从 i+1 到 l
            # 调用优化后的子问题计算
            hist_f_minus_1_l_minus_j = compute_hist_optimized(f - 1, l - j, x, memo)

            # 更新当前的直方图 hist_f_l
            for k in range(f):  # 遍历子问题结果
                if (j - i) < x:  # 跨度小于限制
                    hist_f_l[k + 1] += hist_f_minus_1_l_minus_j[k]
                else:  # 跨度大于等于限制
                    hist_f_l[k] += hist_f_minus_1_l_minus_j[k]

    # 缓存结果
    memo[(f, l)] = hist_f_l
    return hist_f_l


In [ ]:
pi, pi2 = [], []
eplison = 0.7
for i in range(K):
    N_fl = compute_hist_optimized(f[i], l[i], threshold, memo=None)
    pi.append(sum(N_fl[f_hat[i]:])/sum(N_fl))
    for j in range(f_hat[i]+1):
        if sum(N_fl[j:])/sum(N_fl) < eplison:
            pi2.append(sum(N_fl[j:])/sum(N_fl))
            break

In [ ]:
Z = sum(1 for prob in pi if prob < eplison)
print(Z)
EZ = sum(pi2)
print(EZ)

In [ ]:
delta = 0.9
CSR = Z/(EZ+np.sqrt(-K*np.log(delta)/2))
print(CSR)

In [ ]:
CSA = sum(f_hat)/np.sqrt(K)
print(CSA)

In [ ]:
def count_word_occurrences(doc, word):
    return doc.count(word)
x_count, y_count = [], []
for doc in documents:
    x_count.append(count_word_occurrences(doc, word_pairs[0][0]))
    y_count.append(count_word_occurrences(doc, word_pairs[0][1]))
dx = sum(1 for num in x_count if num != 0)
dy = sum(1 for num in y_count if num != 0)
dxy = sum(1 for num in f_hat if num != 0)

In [ ]:
cPMId=dxy/(dx*dy/K+np.sqrt(K)/(2*threshold)*np.sqrt(-np.log(delta)/2))
print(cPMId)

In [ ]:
cPMIz = Z/(dx*dy/K+np.sqrt(K)/(2*threshold)*np.sqrt(-np.log(delta)/2))
print(cPMIz)